In [2]:
import pandas as pd
import numpy as np
import random
from itertools import combinations
import sys
sys.path.append('..')
from pairadigm import Pairadigm, LLMClient

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [3]:
# Load the data
data = pd.read_csv('data/emobank_small_sample.csv')
print(f"Loaded data with {len(data)} sentences")
print(f"Columns: {data.columns.tolist()}")

Loaded data with 30 sentences
Columns: ['id', 'split', 'V', 'A', 'D', 'text']


In [4]:
def simulate_human_annotation(val1, val2, arousal1, arousal2, dom1, dom2, 
                             concept, noise_level=0.3):
    """
    Simulate human annotation based on ground truth values with some noise.
    
    Parameters:
    - val1, val2: valence scores for items 1 and 2
    - arousal1, arousal2: arousal scores for items 1 and 2  
    - dom1, dom2: dominance scores for items 1 and 2
    - concept: which concept to annotate ('valence', 'arousal', 'dominance')
    - noise_level: how much noise to add to decisions (0 = perfect, 1 = random)
    
    Returns:
    - 'Text1' or 'Text2' indicating which text has higher concept value
    """
    
    # Get the relevant scores for the concept
    if concept == 'valence':
        score1, score2 = val1, val2
    elif concept == 'arousal':
        score1, score2 = arousal1, arousal2
    elif concept == 'dominance':
        score1, score2 = dom1, dom2
    else:
        raise ValueError("concept must be 'valence', 'arousal', or 'dominance'")
    
    # Calculate true difference
    true_diff = score1 - score2
    
    # Add noise to the decision
    # The larger the true difference, the less likely we are to make an error
    error_prob = noise_level * np.exp(-abs(true_diff) * 2)  # Exponential decay based on difference
    
    if np.random.random() < error_prob:
        # Make an error - flip the decision
        return 'Text2' if true_diff > 0 else 'Text1'
    else:
        # Make correct decision
        return 'Text1' if true_diff > 0 else 'Text2'

def create_pairwise_dataset(data, num_pairs_per_item=8, num_annotators=3):
    """
    Create a pairwise comparison dataset with simulated human annotations.
    """
    
    # Generate pairings using Pairadigm
    temp_pairadigm = Pairadigm(
        data=data,
        item_id_name='id',
        text_name='text',
        model_name='gpt-4o',  # temporary, just for pairing
        target_concept='valence'  # temporary, just for pairing
    )
    
    # Generate pairings
    pairings_df = temp_pairadigm.pair_items(
        items=data['id'].tolist(),
        num_pairs_per_item=num_pairs_per_item,
        random_seed=42
    )
    
    print(f"Generated {len(pairings_df)} pairs")
    
    # Add text and emotion scores for both items
    id_to_data = data.set_index('id').to_dict('index')
    
    pairwise_data = []
    
    for _, row in pairings_df.iterrows():
        item1_id = row['item1']
        item2_id = row['item2']
        
        item1_data = id_to_data[item1_id]
        item2_data = id_to_data[item2_id]
        
        pair_row = {
            'item1_id': item1_id,
            'item2_id': item2_id,
            'item1_text': item1_data['text'],
            'item2_text': item2_data['text'],
            'item1_valence': item1_data['V'],
            'item2_valence': item2_data['V'],
            'item1_arousal': item1_data['A'],
            'item2_arousal': item2_data['A'],
            'item1_dominance': item1_data['D'],
            'item2_dominance': item2_data['D']
        }
        
        # Generate human annotations for each concept
        for concept in ['valence', 'arousal', 'dominance']:
            for annotator_id in range(1, num_annotators + 1):
                # Each annotator has slightly different noise levels
                noise_level = 0.2 + (annotator_id - 1) * 0.1  # 0.2, 0.3, 0.4
                
                annotation = simulate_human_annotation(
                    item1_data['V'], item2_data['V'],
                    item1_data['A'], item2_data['A'], 
                    item1_data['D'], item2_data['D'],
                    concept=concept,
                    noise_level=noise_level
                )
                
                pair_row[f'{concept}_human_{annotator_id}'] = annotation
        
        pairwise_data.append(pair_row)
    
    return pd.DataFrame(pairwise_data)



In [5]:
# Generate the pairwise dataset
print("Generating pairwise comparisons with simulated human annotations...")
pairwise_df = create_pairwise_dataset(
    data=data,
    num_pairs_per_item=10,  # Each sentence paired with ~8 others
    num_annotators=3       # 3 simulated annotators per concept
)

print(f"Created pairwise dataset with {len(pairwise_df)} pairs")
print(f"Columns: {pairwise_df.columns.tolist()}")

# Show sample of the data
print("\nSample of pairwise data:")
print(pairwise_df[['item1_id', 'item2_id', 'valence_human_1', 'arousal_human_1', 'dominance_human_1']].head())

# Calculate annotation agreement for each concept
def calculate_agreement(df, concept):
    """Calculate inter-annotator agreement for a concept."""
    cols = [f'{concept}_human_1', f'{concept}_human_2', f'{concept}_human_3']
    
    agreements = []
    for _, row in df.iterrows():
        annotations = [row[col] for col in cols]
        # Calculate pairwise agreement
        total_pairs = 0
        agreeing_pairs = 0
        for i in range(len(annotations)):
            for j in range(i+1, len(annotations)):
                total_pairs += 1
                if annotations[i] == annotations[j]:
                    agreeing_pairs += 1
        
        if total_pairs > 0:
            agreements.append(agreeing_pairs / total_pairs)
    
    return np.mean(agreements)

print("\nInter-annotator agreement:")
for concept in ['valence', 'arousal', 'dominance']:
    agreement = calculate_agreement(pairwise_df, concept)
    print(f"{concept.capitalize()}: {agreement:.3f}")

# Save the dataset
output_file = 'data/emobank_small_sample_simAnnotations.csv'
pairwise_df.to_csv(output_file, index=False)
print(f"\nSaved pairwise dataset to {output_file}")

# Create a summary
print(f"\nDataset Summary:")
print(f"- Original sentences: {len(data)}")
print(f"- Pairwise comparisons: {len(pairwise_df)}")
print(f"- Concepts: valence, arousal, dominance")
print(f"- Annotators per concept: 3")
print(f"- Total annotations: {len(pairwise_df) * 3 * 3} (pairs × concepts × annotators)")

# Show distribution of true differences to validate our simulation
print(f"\nTrue score differences (to validate simulation quality):")
for concept in ['valence', 'arousal', 'dominance']:
    col1 = f'item1_{concept}'
    col2 = f'item2_{concept}'
    diffs = abs(pairwise_df[col1] - pairwise_df[col2])
    print(f"{concept.capitalize()} - Mean absolute difference: {diffs.mean():.3f} (std: {diffs.std():.3f})")

Generating pairwise comparisons with simulated human annotations...
Generated 177 pairs
Created pairwise dataset with 177 pairs
Columns: ['item1_id', 'item2_id', 'item1_text', 'item2_text', 'item1_valence', 'item2_valence', 'item1_arousal', 'item2_arousal', 'item1_dominance', 'item2_dominance', 'valence_human_1', 'valence_human_2', 'valence_human_3', 'arousal_human_1', 'arousal_human_2', 'arousal_human_3', 'dominance_human_1', 'dominance_human_2', 'dominance_human_3']

Sample of pairwise data:
                       item1_id                               item2_id  \
0                   SemEval_763           hotel-california_33157_33182   
1                  SemEval_1155  blog-new-year's-resolutions_1573_1737   
2  Nathans_Bylichka_45926_45965        How_soon-Lebron-James_2193_2304   
3         Ant_Robot_19422_19434           Nathans_Bylichka_15318_15334   
4           detroit_12208_12263  blog-new-year's-resolutions_1573_1737   

  valence_human_1 arousal_human_1 dominance_human_1  
0 

/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Personal Projects/pairadigm/pairadigm.py:277: UserWarning: cgcot_prompts must be a non-empty list of prompt templates. Some methods may not work until this is set. You can set the CGCOT prompts using .set_cgcot_prompts()
  warnings.warn("cgcot_prompts must be a non-empty list of prompt templates. Some methods may not work until this is set. You can set the CGCOT prompts using .set_cgcot_prompts()", UserWarning)
